# Gaussian mixture model fit by EM, from scratch (numpy only)

This notebook goes with `notes.md` §11. It fits a full-covariance Gaussian
mixture by EM on 2-D data drawn from three known Gaussians, checks that the
log-likelihood never decreases (the ELBO argument of §11.2), compares the
fitted parameters with the true ones and with sklearn's `GaussianMixture`,
and then checks the k-means connection of §11.4: EM with every covariance
frozen at a tiny $\varepsilon I$ gives exactly the same assignments as Lloyd's
algorithm from the same starting centers.

All randomness comes from fixed seeds, and the notebook runs in a few seconds
on a CPU.

In [1]:
import numpy as np

np.random.seed(42)

## 1. Numerically stable building blocks (notes §11.3)

The E-step needs $\log \sum_j \exp(a_j)$. Computing it directly overflows once
the $a_j$ are large (and underflows to $\log 0$ once they are very negative,
which is the usual case for log-densities). `logsumexp` subtracts the maximum
first: $\log\sum_j e^{a_j} = m + \log\sum_j e^{a_j - m}$ with $m = \max_j a_j$.
`log_gaussian` evaluates $\log \mathcal{N}(x; \mu, \Sigma)$ for every row using
a Cholesky factor $\Sigma = LL^\top$ instead of an explicit inverse. The checks
compare `logsumexp([1000, 1000])` with $1000 + \log 2$, and `log_gaussian`
with scipy's implementation.

In [2]:
def logsumexp(a, axis=None):
    # log(sum(exp(a))) without overflow: subtract the max first.
    m = np.max(a, axis=axis, keepdims=True)
    return (m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))).squeeze(axis)


def log_gaussian(X, mu, Sigma):
    # log N(x; mu, Sigma) for every row of X, via a Cholesky solve.
    d = X.shape[1]
    L = np.linalg.cholesky(Sigma)
    diff = X - mu                                  # (n, d)
    z = np.linalg.solve(L, diff.T)                 # L z = diff^T, shape (d, n)
    maha = np.sum(z ** 2, axis=0)                  # squared Mahalanobis distance
    logdet = 2.0 * np.sum(np.log(np.diag(L)))
    return -0.5 * (d * np.log(2 * np.pi) + logdet + maha)


print("logsumexp([1000, 1000]) =", logsumexp(np.array([1000.0, 1000.0])),
      " expected:", 1000 + np.log(2))
assert np.isclose(logsumexp(np.array([1000.0, 1000.0])), 1000 + np.log(2))

from scipy.stats import multivariate_normal
S_test = np.array([[1.0, 0.6], [0.6, 1.0]])
X_test = np.array([[0.0, 0.0], [1.0, -2.0], [3.0, 3.0]])
ours = log_gaussian(X_test, np.array([0.5, 0.5]), S_test)
ref = multivariate_normal(mean=[0.5, 0.5], cov=S_test).logpdf(X_test)
print("log_gaussian:", np.round(ours, 6), " scipy:", np.round(ref, 6))
assert np.allclose(ours, ref)
print("PASS: both building blocks agree with the reference values.")

logsumexp([1000, 1000]) = 1000.6931471805599  expected: 1000.6931471805599


log_gaussian: [-1.770984 -7.864734 -5.520984]  scipy: [-1.770984 -7.864734 -5.520984]
PASS: both building blocks agree with the reference values.


## 2. Data from three known Gaussians

1500 points are drawn from a mixture with weights $(0.5, 0.3, 0.2)$ and three
elongated, tilted covariances. This is a case where k-means' implicit
assumption of round, equal-size clusters is wrong and a full-covariance
mixture is the correct model.

In [3]:
true_means = np.array([[0.0, 0.0], [5.0, 1.0], [2.0, 5.0]])
true_covs = np.array([
    [[1.0, 0.6], [0.6, 1.0]],       # tilted ellipse
    [[1.2, -0.4], [-0.4, 0.5]],     # tilted the other way
    [[0.4, 0.0], [0.0, 1.5]],       # tall and thin
])
true_pi = np.array([0.5, 0.3, 0.2])
n = 1500
counts = np.random.multinomial(n, true_pi)
X = np.vstack([np.random.multivariate_normal(true_means[j], true_covs[j], counts[j])
               for j in range(3)])
print("points per component:", counts, " X shape:", X.shape)

points per component: [733 458 309]  X shape: (1500, 2)


## 3. EM, and the log-likelihood never decreases (notes §11.2-§11.3)

`gmm_em` alternates the two steps derived in the notes. E-step: compute the
responsibilities $\gamma_{ij}$ by Bayes' rule, entirely in log space. M-step:
set $N_j = \sum_i \gamma_{ij}$, $\pi_j = N_j / n$, $\mu_j$ = the
responsibility-weighted mean, and $\Sigma_j$ = the responsibility-weighted
covariance, plus $10^{-6} I$ so that no covariance collapses onto a single
point (the unbounded-likelihood trap of §11.3). The log-likelihood
$\sum_i \log p(x_i)$ is recorded each iteration. The assert checks that the
trace never decreases, which is the monotonicity proof of §11.2.

The `freeze_cov` option is used in section 6: it holds every covariance fixed
and the weights uniform, so that only the means are learned.

In [4]:
def gmm_em(X, k, n_iters=200, tol=1e-8, seed=0, init_means=None,
           freeze_cov=None, reg_covar=1e-6):
    # Returns (pi, means, covs, ll_trace, gamma).
    n, d = X.shape
    rng = np.random.default_rng(seed)
    if init_means is None:
        means = X[rng.choice(n, size=k, replace=False)].copy()
    else:
        means = init_means.copy()
    covs = np.array([np.cov(X.T) + reg_covar * np.eye(d) for _ in range(k)])
    if freeze_cov is not None:
        covs = np.array([freeze_cov.copy() for _ in range(k)])
    pi = np.full(k, 1.0 / k)

    ll_trace = []
    gamma = None
    for _ in range(n_iters):
        # E-step in log space: log_joint[i, j] = log pi_j + log N(x_i; mu_j, Sigma_j)
        log_joint = np.stack(
            [np.log(pi[j]) + log_gaussian(X, means[j], covs[j]) for j in range(k)],
            axis=1)                                  # (n, k)
        log_norm = logsumexp(log_joint, axis=1)      # log p(x_i)
        gamma = np.exp(log_joint - log_norm[:, None])

        ll_trace.append(float(np.sum(log_norm)))
        if len(ll_trace) > 1 and abs(ll_trace[-1] - ll_trace[-2]) < tol:
            break

        # M-step: closed forms from notes §11.3
        N = gamma.sum(axis=0)                        # effective counts
        means = (gamma.T @ X) / N[:, None]           # weighted means
        if freeze_cov is None:
            pi = N / n
            for j in range(k):
                diff = X - means[j]
                covs[j] = (gamma[:, j, None] * diff).T @ diff / N[j]
                covs[j] += reg_covar * np.eye(d)
        # with freeze_cov, covariances and the uniform weights stay fixed
    return pi, means, covs, ll_trace, gamma


pi, means, covs, ll_trace, gamma = gmm_em(X, k=3, seed=3)

print(f"converged in {len(ll_trace)} EM iterations; log-likelihood (every 5th):")
for i in range(0, len(ll_trace), 5):
    print(f"  iter {i:3d}: LL = {ll_trace[i]:14.4f}")
print(f"  iter {len(ll_trace) - 1:3d}: LL = {ll_trace[-1]:14.4f}  (final)")
assert np.all(np.diff(ll_trace) >= -1e-7), "log-likelihood decreased"
print("PASS: log-likelihood never decreases (notes §11.2).")

converged in 24 EM iterations; log-likelihood (every 5th):
  iter   0: LL =     -7071.4068
  iter   5: LL =     -5841.9738
  iter  10: LL =     -5358.2899
  iter  15: LL =     -5293.3464
  iter  20: LL =     -5293.3419
  iter  23: LL =     -5293.3419  (final)
PASS: log-likelihood never decreases (notes §11.2).


## 4. Recovered vs. true parameters

EM does not know which fitted component corresponds to which true one, so the
components are matched greedily by nearest mean. With 1500 points, every
recovered mean should be within 0.25 of its true mean, and the weights should
be close to $(0.5, 0.3, 0.2)$.

In [5]:
def match_components(est_means, true_means):
    # Greedy matching of estimated components to true ones (k is small).
    k = len(true_means)
    remaining = list(range(k))
    order = np.zeros(k, dtype=int)
    for t in range(k):
        dists = [np.linalg.norm(est_means[j] - true_means[t]) for j in remaining]
        pick = remaining[int(np.argmin(dists))]
        order[t] = pick
        remaining.remove(pick)
    return order


order = match_components(means, true_means)
for t in range(3):
    j = order[t]
    print(f"  comp {t}: true mean {np.round(true_means[t], 3)}  est mean {np.round(means[j], 3)}"
          f"  | true pi {true_pi[t]:.3f}  est pi {pi[j]:.3f}")
mean_err = max(np.linalg.norm(means[order[t]] - true_means[t]) for t in range(3))
print(f"max mean error: {mean_err:.4f}")
assert mean_err < 0.25, "recovered means too far from truth"
print("PASS: recovered means within 0.25 of the generating means.")

  comp 0: true mean [0. 0.]  est mean [-0.056 -0.026]  | true pi 0.500  est pi 0.490
  comp 1: true mean [5. 1.]  est mean [5.07  1.021]  | true pi 0.300  est pi 0.305
  comp 2: true mean [2. 5.]  est mean [2.009 5.054]  | true pi 0.200  est pi 0.206
max mean error: 0.0732
PASS: recovered means within 0.25 of the generating means.


## 5. Check against sklearn's GaussianMixture

sklearn fits the same model by EM (best of 5 initializations). Both fits
should reach the same optimum: means within 0.1 of each other, and our final
log-likelihood no more than 1 below sklearn's. sklearn reports the mean
log-likelihood per point, so it is multiplied by $n$.

In [6]:
from sklearn.mixture import GaussianMixture

sk = GaussianMixture(n_components=3, covariance_type="full",
                     random_state=0, n_init=5).fit(X)
sk_order = match_components(sk.means_, true_means)
sk_ll = sk.score(X) * len(X)
print(f"  ours    final LL = {ll_trace[-1]:14.4f}")
print(f"  sklearn final LL = {sk_ll:14.4f}")
for t in range(3):
    print(f"  comp {t}: ours {np.round(means[order[t]], 3)}  "
          f"sklearn {np.round(sk.means_[sk_order[t]], 3)}")
max_gap = max(np.linalg.norm(means[order[t]] - sk.means_[sk_order[t]]) for t in range(3))
print(f"  max |ours - sklearn| over means: {max_gap:.4f}")
assert max_gap < 0.1, "our EM disagrees with sklearn's fit"
assert ll_trace[-1] >= sk_ll - 1.0, "our LL notably below sklearn's"
print("PASS: matches sklearn (same optimum up to tolerance).")

  ours    final LL =     -5293.3419
  sklearn final LL =     -5293.3512
  comp 0: ours [-0.056 -0.026]  sklearn [-0.058 -0.028]
  comp 1: ours [5.07  1.021]  sklearn [5.07  1.021]
  comp 2: ours [2.009 5.054]  sklearn [2.009 5.049]
  max |ours - sklearn| over means: 0.0051
PASS: matches sklearn (same optimum up to tolerance).


## 6. k-means is EM with tiny frozen spherical covariances (notes §11.4)

Freeze every covariance at $\varepsilon I$ with $\varepsilon = 10^{-3}$ and the
weights at $1/3$. The responsibilities become a softmax of
$-\|x_i - \mu_j\|^2 / (2\varepsilon)$, which for small $\varepsilon$ puts
essentially all weight on the nearest center, so the E-step becomes the
k-means assignment step and the M-step becomes the k-means mean update.
Running this EM and plain Lloyd's algorithm from the same three starting
centers should give identical assignments for all 1500 points, and the
smallest "largest responsibility" over all points should be 1 to many
decimal places.

In [7]:
def lloyds_kmeans(X, init_centers, n_iters=200):
    # Plain Lloyd's algorithm from given initial centers.
    centers = init_centers.copy()
    labels = None
    for _ in range(n_iters):
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        new_labels = d2.argmin(axis=1)
        for j in range(len(centers)):
            mask = new_labels == j
            if mask.any():
                centers[j] = X[mask].mean(axis=0)
        if labels is not None and np.array_equal(labels, new_labels):
            break
        labels = new_labels
    return centers, labels


d = X.shape[1]
eps = 1e-3
rng = np.random.default_rng(11)
init = X[rng.choice(len(X), size=3, replace=False)].copy()

_, em_means, _, _, gamma_hard = gmm_em(
    X, k=3, init_means=init, freeze_cov=eps * np.eye(d), reg_covar=0.0)
em_labels = gamma_hard.argmax(axis=1)
hardness = gamma_hard.max(axis=1).min()
print(f"frozen-covariance EM, eps = {eps}: min over points of max responsibility "
      f"= {hardness:.6f}")

km_centers, km_labels = lloyds_kmeans(X, init)
agree = float(np.mean(em_labels == km_labels))
print(f"assignment agreement with Lloyd's from the same init: {agree:.4%}")
assert agree == 1.0, "frozen-covariance EM did not reproduce k-means"
print("PASS: hard-EM assignments are identical to k-means assignments.")
print("center gap (EM weighted means vs. k-means means):",
      np.round(np.linalg.norm(em_means - km_centers, axis=1), 8))

frozen-covariance EM, eps = 0.001: min over points of max responsibility = 1.000000
assignment agreement with Lloyd's from the same init: 100.0000%
PASS: hard-EM assignments are identical to k-means assignments.
center gap (EM weighted means vs. k-means means): [0. 0. 0.]
